In [2]:
import glob

import numpy as np
import pandas as pd
from model.ctabgan import CTABGAN
from model.eval.evaluation import get_utility_metrics, privacy_metrics, stat_sim

In [3]:
num_exp = 5
dataset = "MimicIV"
real_path = "Real_Datasets/MimicIV_clean.csv"
fake_file_root = "Fake_Datasets"

In [ ]:
synthesizer = CTABGAN(
    raw_csv_path=real_path,
    test_ratio=0.20,
    categorical_columns=["gender", "hospital_expire_flag"],
    log_columns=[],  # none unless you want to log-transform skewed labs
    mixed_columns={},  # none unless you confirm heavy zero-mass columns
    general_columns=[
        "heart_rate_min",
        "heart_rate_max",
        "heart_rate_mean",
        "sbp_min",
        "sbp_max",
        "sbp_mean",
        "dbp_min",
        "dbp_max",
        "dbp_mean",
        "resp_rate_min",
        "resp_rate_max",
        "resp_rate_mean",
        "temperature_min",
        "temperature_max",
        "temperature_mean",
        "spo2_min",
        "spo2_max",
        "spo2_mean",
        "creatinine_min",
        "creatinine_max",
        "sodium_min",
        "sodium_max",
        "potassium_min",
        "potassium_max",
        "hemoglobin_min",
        "hemoglobin_max",
        "wbc_min",
        "wbc_max",
    ],
    non_categorical_columns=[],  # since we dropped IDs
    integer_columns=["age_at_intime"],
    problem_type={"Classification": "hospital_expire_flag"},
    class_dim=(256, 256),
    random_dim=100,
    num_channels=64,
    l2scale=1e-5,
    batch_size=512,
    epochs=150,
)

for i in range(num_exp):
    synthesizer.fit()
    syn = synthesizer.generate_samples()
    syn.to_csv(
        fake_file_root
        + "/"
        + dataset
        + "/"
        + dataset
        + "_fake_{exp}.csv".format(exp=i),
        index=False,
    )

 15%|█▍        | 22/150 [01:13<07:06,  3.33s/it]

In [4]:
fake_paths = glob.glob(fake_file_root + "/" + dataset + "/" + "*")

In [5]:
def make_numeric_for_utility(path_in, path_out, target="hospital_expire_flag"):
    df = pd.read_csv(path_in)

    # Ensure target is last or separated (depending on your eval code)
    y = df[target]
    X = df.drop(columns=[target])

    # One-hot encode any object/categorical columns (e.g., gender)
    X_enc = pd.get_dummies(X, drop_first=False)

    out = pd.concat([X_enc, y], axis=1)
    out.to_csv(path_out, index=False)
    return path_out


real_path_u = make_numeric_for_utility(real_path, "Real_Datasets/MimicIVU.csv")

fake_paths_u = []
for i, fp in enumerate(fake_paths):
    fake_paths_u.append(
        make_numeric_for_utility(fp, f"Fake_Datasets/MimicIV/MimicIV_fakeU_{i}.csv")
    )

    # after generating real one-hot columns, align fake to real:
real_cols = pd.read_csv(real_path_u).columns


def align_to_real(path_in, real_cols, target):
    df = pd.read_csv(path_in)
    y = df[target]
    X = pd.get_dummies(df.drop(columns=[target]), drop_first=False)

    X = X.reindex(columns=[c for c in real_cols if c != target], fill_value=0)
    out = pd.concat([X, y], axis=1)
    return out

In [6]:
import pandas as pd

df = pd.read_csv(fake_file_root + "/" + dataset + "/" + dataset + "_fakeU_0.csv")
df.info()
df.head()
df.tail()
df.isna().sum()
df.replace([np.inf, -np.inf], np.nan).isna().sum()

<class 'pandas.DataFrame'>
RangeIndex: 23698 entries, 0 to 23697
Data columns (total 32 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   age_at_intime         23698 non-null  int64  
 1   heart_rate_min        23698 non-null  float64
 2   heart_rate_max        23698 non-null  float64
 3   heart_rate_mean       23698 non-null  float64
 4   sbp_min               23698 non-null  float64
 5   sbp_max               23698 non-null  float64
 6   sbp_mean              23698 non-null  float64
 7   dbp_min               23698 non-null  float64
 8   dbp_max               23698 non-null  float64
 9   dbp_mean              23698 non-null  float64
 10  resp_rate_min         23698 non-null  float64
 11  resp_rate_max         23698 non-null  float64
 12  resp_rate_mean        23698 non-null  float64
 13  temperature_min       23698 non-null  float64
 14  temperature_max       23698 non-null  float64
 15  temperature_mean      23698 no

age_at_intime           0
heart_rate_min          0
heart_rate_max          0
heart_rate_mean         0
sbp_min                 0
sbp_max                 0
sbp_mean                0
dbp_min                 0
dbp_max                 0
dbp_mean                0
resp_rate_min           0
resp_rate_max           0
resp_rate_mean          0
temperature_min         0
temperature_max         0
temperature_mean        0
spo2_min                0
spo2_max                0
spo2_mean               0
creatinine_min          0
creatinine_max          0
sodium_min              0
sodium_max              0
potassium_min           0
potassium_max           0
hemoglobin_min          0
hemoglobin_max          0
wbc_min                 0
wbc_max                 0
gender_F                0
gender_M                0
hospital_expire_flag    0
dtype: int64

In [ ]:
model_dict = {"Classification": ["lr", "dt", "rf", "mlp", "svm"]}
result_mat = get_utility_metrics(
    real_path, fake_paths, "MinMax", model_dict, test_ratio=0.20
)

result_df = pd.DataFrame(result_mat, columns=["Acc", "AUC", "F1_Score"])
result_df.index = list(model_dict.values())[0]
result_df

In [9]:
mimic_categorical = ["hospital_expire_flag", "gender"]
stat_res_avg = []
for fake_path in fake_paths:
    stat_res = stat_sim(real_path, fake_path, mimic_categorical)
    stat_res_avg.append(stat_res)

stat_columns = [
    "Average WD (Continuous Columns",
    "Average JSD (Categorical Columns)",
    "Correlation Distance",
]
stat_results = pd.DataFrame(
    np.array(stat_res_avg).mean(axis=0).reshape(1, 3), columns=stat_columns
)
stat_results

column:  gender JSD:  0.0023714533747027353
column:  age_at_intime WD:  0.01244177020940734
column:  heart_rate_min WD:  0.030586353974476822
column:  heart_rate_max WD:  0.010260060627542785
column:  heart_rate_mean WD:  0.01896235187782286
column:  sbp_min WD:  0.009850476178150139
column:  sbp_max WD:  0.008385911051539963
column:  sbp_mean WD:  0.006932519721692313
column:  dbp_min WD:  0.0037966987158353518
column:  dbp_max WD:  0.005572619229612972
column:  dbp_mean WD:  0.0037609176354380064
column:  resp_rate_min WD:  0.012317056351298771
column:  resp_rate_max WD:  0.007680377602566441
column:  resp_rate_mean WD:  0.004384642195124851
column:  temperature_min WD:  0.010561833505713955
column:  temperature_max WD:  0.0069327949976576795
column:  temperature_mean WD:  0.00357602120737053
column:  spo2_min WD:  0.005355377179137384
column:  spo2_max WD:  0.007908280927989895
column:  spo2_mean WD:  0.006406039869986791
column:  creatinine_min WD:  0.00450104623452687
column:  cre

KeyError: 'gender'

In [7]:
priv_res_avg = []
for fake_path in fake_paths_u:
    priv_res = privacy_metrics(real_path_u, fake_path)
    priv_res_avg.append(priv_res)

privacy_columns = [
    "DCR between Real and Fake (5th perc)",
    "DCR within Real(5th perc)",
    "DCR within Fake (5th perc)",
    "NNDR between Real and Fake (5th perc)",
    "NNDR within Real (5th perc)",
    "NNDR within Fake (5th perc)",
]
privacy_results = pd.DataFrame(
    np.array(priv_res_avg).mean(axis=0).reshape(1, 6), columns=privacy_columns
)
privacy_results

,DCR between Real and Fake (5th perc),DCR within Real(5th perc),DCR within Fake (5th perc),NNDR between Real and Fake (5th perc),NNDR within Real (5th perc),NNDR within Fake (5th perc)
0,2.37254,2.219882,2.263921,0.831381,0.817386,0.812398


In [7]:
import os

import pandas as pd
from Evaluation.cdf_tail_metrics import CDFTailMetrics
from Evaluation.rare_event_recall import RareEventRecall
from Evaluation.support_coverage import SupportCoverage

n_experiments = 5

cdf_eval = CDFTailMetrics(label_col="hospital_expire_flag", tau=0.9)
sc_eval = SupportCoverage(
    label_col="hospital_expire_flag",
    k=2,  # pairwise support coverage
    n_bins=10,
    rare_threshold=0.05,
    max_subsets=100,  # sample up to 100 feature pairs
    include_label_in_combo=True,
    random_state=42,
)
rer_eval = RareEventRecall(label_col="hospital_expire_flag")

rows = []

for exp in range(n_experiments):
    syn_path = os.path.join(fake_file_root, dataset, f"{dataset}_fakeU_{exp}.csv")
    if not os.path.exists(syn_path):
        print("Missing:", syn_path)
        continue

    res = {"exp": exp, "syn_path": syn_path}
    res.update(cdf_eval.evaluate_paths(real_path_u, syn_path))
    res.update(sc_eval.evaluate_paths(real_path_u, syn_path))
    res.update(rer_eval.evaluate_paths(real_path_u, syn_path))
    rows.append(res)

results_df = pd.DataFrame(rows)
results_df.to_csv("artifacts/MimicEval.csv")
results_df

,exp,syn_path,cdf_tail_div_age_at_intime,cdf_tail_div_heart_rate_min,cdf_tail_div_heart_rate_max,cdf_tail_div_heart_rate_mean,cdf_tail_div_sbp_min,cdf_tail_div_sbp_max,cdf_tail_div_sbp_mean,cdf_tail_div_dbp_min,...,avg_num_syn_combos,rare_class,rare_event_recall_real_baseline,rare_event_total_real_baseline,rare_event_correct_real_baseline,rare_event_recall_synthetic,rare_event_total_real,rare_event_correct_synthetic,rare_class_count_real,n_classes_real
0,0,Fake_Datasets/MimicIV/MimicIV_fakeU_0.csv,0.010085,0.051988,0.015149,0.013967,0.020635,0.012491,0.018651,0.009959,...,176.57,1.0,0.219844,1028,226,0.184526,3425,632,3425,2
1,1,Fake_Datasets/MimicIV/MimicIV_fakeU_1.csv,0.012406,0.022702,0.049287,0.045531,0.062453,0.039708,0.032070,0.027260,...,175.28,1.0,0.219844,1028,226,0.246423,3425,844,3425,2
2,2,Fake_Datasets/MimicIV/MimicIV_fakeU_2.csv,0.009537,0.018061,0.004473,0.006203,0.010001,0.011942,0.007385,0.024939,...,176.65,1.0,0.219844,1028,226,0.216058,3425,740,3425,2
3,3,Fake_Datasets/MimicIV/MimicIV_fakeU_3.csv,0.012406,0.022702,0.049287,0.045531,0.062453,0.039708,0.032070,0.027260,...,175.28,1.0,0.219844,1028,226,0.246423,3425,844,3425,2
4,4,Fake_Datasets/MimicIV/MimicIV_fakeU_4.csv,0.010085,0.051988,0.015149,0.013967,0.020635,0.012491,0.018651,0.009959,...,176.57,1.0,0.219844,1028,226,0.184526,3425,632,3425,2


In [10]:
from Evaluation.detector_treeshap import DetectorTreeSHAP

xai_detector = DetectorTreeSHAP(
    test_size=0.3,
    random_state=42,
    n_estimators=300,
    output_dir=f"artifacts/xai_detector/{dataset}",
)

detector_results = xai_detector.evaluate_paths(real_path, syn_path)
detector_results

{'detector_auc': 0.9999993767928336,
 'detector_accuracy': 0.9991560587945706,
 'detector_average_precision': 0.9999993655639845,
 'n_real': 23698,
 'n_synthetic': 23698,
 'n_features_original': 30,
 'n_features_encoded': 30}